# 06 - Conclusions

Section **3.6 Conclusions** of the report.

## Brief

Synthesis of results, lessons learned, and possible directions for future work.

- Synthesis of results.
- Lessons learned.
- Future work.

In [1]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import load_dotenv
from scipy import sparse

from diplo_mod_1.constants import (
    CONFIGS,
    INTERIM,
    MODELS,
    PRIMARY_CSV,
    PROCESSED,
    RANDOM_STATE,
    RAW,
    REPORTS,
)
from diplo_mod_1.preprocessing.cleaner import DataCleaner
from diplo_mod_1.preprocessing.encoders import TabularEncoder
from diplo_mod_1.preprocessing.feature_engineer import FeatureEngineer
from diplo_mod_1.training.config import TuningHistory
from diplo_mod_1.training.nn_model import WineScoreNet, WineScorePredictorNet

load_dotenv(override=True)

# Experiment tracking is opt-in: set WANDB_ENABLED=true (in .env or the shell)
# to fetch/export from Weights & Biases. Off by default so `poe check` / nbmake
# executions don't require W&B credentials just to open this notebook.
WANDB_ENABLED = os.environ.get("WANDB_ENABLED", "false").lower() == "true"

## Step 1 — Synthesis of results

The final numbers already on record across notebooks 03, 04, 04b, and 05, pulled together in one place.

In [2]:
xgb_history = TuningHistory.model_validate_json(
    (REPORTS / "xgboost_metrics.json").read_text(encoding="utf-8")
)
nn_history = TuningHistory.model_validate_json(
    (REPORTS / "nn_metrics.json").read_text(encoding="utf-8")
)
nn_ablation_history = TuningHistory.model_validate_json(
    (REPORTS / "nn_tabular_ablation_metrics.json").read_text(encoding="utf-8")
)


def _test_metrics(run):
    m = next(x for x in run.metrics if x.split == "test")
    return m.rmse, m.r2


rows = []
for label, history in [
    ("XGBoost (44 tabular + 2000 TF-IDF, tuned)", xgb_history),
    ("Neural Net (44 tabular + 2000 TF-IDF, tuned)", nn_history),
    ("Neural Net (44 tabular only, same tuning budget)", nn_ablation_history),
]:
    best = history.best_run(split="test")
    assert best is not None
    rmse, r2 = _test_metrics(best)
    rows.append({"model": label, "test_rmse": rmse, "test_r2": r2})

summary_df = pd.DataFrame(rows)
summary_df

,model,test_rmse,test_r2
0,"XGBoost (44 tabular + 2000 TF-IDF, tuned)",1.445316,0.774511
1,"Neural Net (44 tabular + 2000 TF-IDF, tuned)",1.462726,0.769046
2,"Neural Net (44 tabular only, same tuning budget)",1.869429,0.622760


Three rounds of NN tuning, all on the same full feature set — each gain came from search/regularization changes, not a different model:

| round | config | test RMSE | test R² |
|---|---|---|---|
| Initial search (10 trials) | `nn_training.json` | 1.595 | 0.725 |
| Dense preload + Optuna pruning (20 trials) | `nn_training_wide.json` | 1.479 | 0.764 |
| + activation choice, grad clipping, LR scheduler (30 trials) | `nn_training_wide_v1.json` | 1.463 | 0.769 |

## Step 2 — What the dataset itself limits

Every improvement attempted in this project — wider search, more capable activations, the TF-IDF block, hyperparameter tuning — worked on the same underlying data. Some real limits live in that data itself, not in either model, and no amount of further tuning removes them. Grounded in `notebooks/01-eda.ipynb`'s own findings, with a "what would change if this changed" angle on each:

**Taster bias.** A one-way ANOVA on `points` by `taster_name` gives F=612.05, p≈0, η²=0.099 — reviewer identity alone explains ~10% of the variance in the target. Per-taster means range from 85.86 to 90.56. `taster_avg_points`/`taster_strictness` already correct for a taster's *average* leniency via target encoding, but a given taster's idiosyncrasy on a *specific* review — beyond their own average — isn't modeled and can't be, from a single score per wine. If this dataset had multiple independent tasters scoring the same wines (averaged), that 10% noise floor would shrink directly, and both models would likely close more of the remaining ~0.22-0.23 unexplained-variance gap (1 − R²) than any further architecture or search change could.

**Price.** 6.9% of rows are missing `price` (median-imputed by (country, variety) group, falling back to the global median); the raw distribution has skewness 18.0 and a tail out to $3,300. `log_price` and `price_vs_variety` are consistently among the highest-SHAP-importance features for *both* models (notebook 05 Step 6) — they're doing real work. Cleaner, more complete price data (fewer imputed values, verified retail rather than listed price) would sharpen the single strongest price-derived signal in the whole feature set.

**`region_2`.** 61.1% missing overall, and structurally so — only present for US wines at all, and even there mostly not recoverable from `region_1` (only 368 of 58,213 candidate rows had a clean 1-to-1 mapping). Dropped entirely from the pipeline (`cleaner.py:44`). More consistent sub-region/appellation labeling across *all* countries, not just some US wines, would add a real geographic signal below `region_1`/`province` — both of which already rank highly in SHAP as it stands.

**Vintage survivorship bias.** Pre-1980 vintages score above the dataset average — not because those wines are better, but because only exceptional old bottles are still around (and worth reviewing) decades later. Notebook 01 calls this out explicitly as survivorship bias, not wine chemistry. `wine_age` as currently computed can't distinguish "old and good" from "old because it survived." A dataset with review timing normalized relative to release date (e.g. "reviewed within N years of vintage" for every row, not just recent ones) would remove this confound rather than encode it.

**Long-tail categories.** `winery` alone has 16,757 unique values, most with only a handful of reviews. `winery_freq` and `winery_avg_points` — both used by both models — are exactly the features noisiest for rare categories: a target-encoded average from 2 reviews is a much weaker estimate than one from 200. This also shows up at the country level: Chile (mean 86.49) and Argentina (86.71) score lower than the US/France and have proportionally fewer reviews. More reviews per winery/variety/country — especially for the underrepresented ones — would stabilize the aggregate-encoded features both models lean on most, not just add raw row count.

**Single-source snapshot.** This entire dataset is one publication's (Wine Enthusiast) house style of tasting notes. The curated `TASTING_KEYWORDS` vocabulary and the TF-IDF vectorizer are both fit to that specific writing style ("tannins", "finish", "palate", "nose" — professional tasting-note language, not how a typical consumer would describe a wine). A broader mix of review sources would be a genuine test of whether the text signal generalizes, or is partly an artifact of one publication's vocabulary.

## Step 3 — XGBoost vs. Neural Network: which, and why

**XGBoost is the better model here** — test R² 0.7745 vs. the NN's 0.7690 (test RMSE 1.445 vs. 1.463), and it stayed narrowly ahead through every NN improvement this project tried: dense-data preloading, a wider Optuna search, activation choice, gradient clipping, an LR scheduler. That's the expected outcome for this dataset — ~130k rows of mixed tabular + sparse bag-of-words features is squarely gradient-boosted-tree territory, and GBMs are well known to be hard to beat on structured/tabular data at this scale, especially against a from-scratch MLP with no sequence-aware text modeling.

**But the neural network doesn't perform badly** — 0.769 vs. 0.775 is a ~1.2% relative RMSE gap, not a blowout, and it closed most of that gap over the course of this project without ever changing its feature set or basic architecture family.

**The more consistent finding across this whole project, for *both* models: what actually moved accuracy was the full 2044-column feature set plus hyperparameter tuning — not model architecture choice.** The evidence is already on record, three separate times:

1. **XGBoost's own tabular-only → tabular+TF-IDF jump**: ~10 separate Optuna searches on the 44 tabular columns alone plateaued at test R² ≈ 0.71-0.73 regardless of depth/regularization/learning-rate changes; adding the same 2000-term TF-IDF block the NN uses pushed it to 0.775 in one step.
2. **The NN's tabular-only ablation** (notebook 04b, same tuning budget, same search space, only 44 columns instead of 2044): test R² 0.623 — worse than even the NN's *first, least-tuned* full-feature run (0.725).
3. **The NN's three-round tuning arc** (Step 1's table above): every gain — 0.725 → 0.764 → 0.769 — came from search/regularization/optimizer changes on the exact same feature set and the same basic MLP family. It never needed a structurally different model to close most of its gap with XGBoost; it needed a wider search and a training loop that could actually converge well (dense data loading, pruning, a scheduler).

Put plainly: neither model's default, untuned hyperparameters got anywhere near either model's final tuned numbers. On this dataset, feature richness and tuning budget mattered more than which of these two model families you picked.

## Step 4 — Predict a new wine (synthetic example)

Re-fits `DataCleaner`/`FeatureEngineer` fresh on the full raw dataset and `TabularEncoder` fresh on the train split — all three are deterministic fit/transform classes given the same data and this project's fixed `RANDOM_STATE`, so re-fitting reproduces exactly what produced `data/processed/` the first time; nothing new is invented here, just orchestrating existing, already-tested classes on a brand new row. `scaler.joblib`/`tfidf_vectorizer.joblib` are loaded directly (already-established pattern, no re-fit needed).

In [3]:
cleaner = DataCleaner()
raw_df = DataCleaner.load(RAW / PRIMARY_CSV)
cleaner.fit(raw_df)

engineer = FeatureEngineer()
engineer.fit(raw_df)

split_idx = np.load(PROCESSED / "split_indices.npz")
featured_full = pd.read_parquet(INTERIM / "02_features.parquet")
train_df = featured_full.iloc[split_idx["train"]]
y_train = np.load(PROCESSED / "nn" / "y_train.npy")

tabular_encoder = TabularEncoder()
tabular_encoder.fit(train_df, y_train)

nn_dir = PROCESSED / "nn"
scaler = joblib.load(nn_dir / "scaler.joblib")
tfidf_vectorizer = joblib.load(nn_dir / "tfidf_vectorizer.joblib")

expected_feature_names = json.loads((nn_dir / "feature_names.json").read_text(encoding="utf-8"))[
    "feature_names"
]
continuous_idx = json.loads((nn_dir / "continuous_column_indices.json").read_text(encoding="utf-8"))

print("Re-fitted DataCleaner, FeatureEngineer, and TabularEncoder on-demand.")

Re-fitted DataCleaner, FeatureEngineer, and TabularEncoder on-demand.


Load both best models (same pattern as notebook 05).

In [4]:
xgb_model = joblib.load(MODELS / "xgboost_best.joblib")

checkpoint = torch.load(MODELS / "nn_best.pt", map_location="cpu")
# nn_best.pt predates NNModelRegistry persisting `activation` -- known from
# this session's run to be 'gelu'; new checkpoints self-describe correctly.
checkpoint_activation = checkpoint.get("activation", "gelu")
nn_net = WineScoreNet(
    input_dim=checkpoint["input_dim"],
    hidden_sizes=checkpoint["hidden_sizes"],
    dropout=checkpoint["dropout"],
    activation=checkpoint_activation,
)
nn_net.load_state_dict(checkpoint["state_dict"])
nn_net.eval()

nn_model = WineScorePredictorNet(
    input_dim=checkpoint["input_dim"],
    hidden_sizes=checkpoint["hidden_sizes"],
    dropout=checkpoint["dropout"],
    activation=checkpoint_activation,
    device="cpu",
)
nn_model.model_ = nn_net

print(f"XGBoost n_features_in_={xgb_model.n_features_in_}, NN activation={checkpoint_activation}")

XGBoost n_features_in_=2044, NN activation=gelu


The transform: `clean` → `engineer` → `tabular_encoder` (same three-call pipeline notebook 02 uses at training time), then vectorize the description and assemble each model's expected input shape (XGBoost: unscaled tabular + TF-IDF; NN: scaled tabular + TF-IDF, densified).

In [5]:
def raw_wine_to_model_inputs(wine: dict):
    """Turn one raw wine record into (xgb_input, nn_input), matching each
    model's expected 2044-column representation exactly.

    `wine` needs the raw CSV schema: country, description, designation,
    price, province, region_1, taster_name, title (must contain a 4-digit
    year, e.g. "Some Wine 2021" -- used to derive wine_age), variety, winery.
    `price`/`designation`/`region_1`/`taster_name` may be None/omitted; they
    fall back the same way training data with missing values did.
    """
    row = pd.DataFrame([wine])
    cleaned = cleaner.clean(row)
    featured = engineer.transform(cleaned)
    tab = tabular_encoder.transform(featured)
    assert tab.feature_names == expected_feature_names, (
        "re-fitted TabularEncoder's column order doesn't match feature_names.json "
        "-- something upstream has drifted from notebook 02's pipeline"
    )

    txt_row = tfidf_vectorizer.transform(featured["description"].astype(str))

    xgb_input = sparse.hstack([sparse.csr_matrix(tab.X), txt_row], format="csr")

    tab_scaled = tab.X.copy()
    tab_scaled[:, continuous_idx] = scaler.transform(tab.X[:, continuous_idx])
    nn_input = (
        sparse.hstack([sparse.csr_matrix(tab_scaled), txt_row], format="csr")
        .toarray()
        .astype(np.float32)
    )

    return xgb_input, nn_input


def predict_wine(wine: dict) -> None:
    xgb_input, nn_input = raw_wine_to_model_inputs(wine)
    xgb_pred = xgb_model.predict(xgb_input)[0]
    nn_pred = nn_model.predict(nn_input)[0]
    print(f"{wine['winery']} {wine['variety']} ({wine['country']}), ${wine['price']}")
    print(f"  XGBoost predicted points: {xgb_pred:.1f}")
    print(f"  Neural Net predicted points: {nn_pred:.1f}")
    print()

Three synthetic examples spanning the price/quality range.

In [6]:
everyday_wine = {
    "country": "US",
    "description": (
        "A simple, easy-drinking red with light cherry and berry notes and a "
        "soft, short finish. Nothing fancy, decent for the price."
    ),
    "designation": None,
    "price": 12.0,
    "province": "California",
    "region_1": "Central Valley",
    "taster_name": "Unknown",
    "title": "Example Everyday Red 2021",
    "variety": "Merlot",
    "winery": "Sunny Valley Cellars",
}

mid_range_wine = {
    "country": "Argentina",
    "description": (
        "A rich, ripe Malbec with dark berry and cherry notes, hints of black "
        "pepper and vanilla oak, medium tannins and a smooth, lasting finish."
    ),
    "designation": "Reserva",
    "price": 28.0,
    "province": "Mendoza Province",
    "region_1": "Lujan de Cuyo",
    "taster_name": "Michael Schachner",
    "title": "Example Winery Malbec Reserva 2019",
    "variety": "Malbec",
    "winery": "Example Winery",
}

luxury_wine = {
    "country": "France",
    "description": (
        "An exceptional, complex wine with layers of dark fruit, spice, "
        "graphite and violet, remarkably concentrated yet elegant, with a "
        "long, polished, structured finish. Aromas of cassis and cedar "
        "linger on the nose. A profound, age-worthy bottling."
    ),
    "designation": "Grand Cru",
    "price": 250.0,
    "province": "Bordeaux",
    "region_1": "Pauillac",
    "taster_name": "Roger Voss",
    "title": "Example Chateau Grand Cru 2016",
    "variety": "Bordeaux-style Red Blend",
    "winery": "Example Chateau",
}

for wine in (everyday_wine, mid_range_wine, luxury_wine):
    predict_wine(wine)

Sunny Valley Cellars Merlot (US), $12.0
  XGBoost predicted points: 83.6
  Neural Net predicted points: 84.9

Example Winery Malbec (Argentina), $28.0
  XGBoost predicted points: 86.8
  Neural Net predicted points: 87.7

Example Chateau Bordeaux-style Red Blend (France), $250.0
  XGBoost predicted points: 94.6
  Neural Net predicted points: 91.8



c:\Users\leona\source\repos\diplo-mod-1\.venv\lib\site-packages\xgboost\core.py:751: UserWarning: [22:46:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Edit freely and re-run — any country/variety/winery/taster not seen during training falls back gracefully (target encoders return their fitted global mean for unseen categories; frequency maps return 0).

In [7]:
your_wine = {
    "country": "Argentina",
    "description": "An exceptional, complex wine with layers of dark fruit, spice, "
    "graphite and violet, remarkably concentrated yet elegant, with a "
    "long, polished, structured finish. Aromas of cassis and cedar "
    "linger on the nose. A profound, age-worthy bottling.",
    "designation": "Mendoza Province",
    "price": 25.0,
    "province": "UCO Valley",
    "region_1": None,
    "taster_name": "",
    "title": "El Enemigo 2018",
    "variety": "Cabernet Franc",
    "winery": "Vigil",
}

predict_wine(your_wine)

Vigil Cabernet Franc (Argentina), $25.0
  XGBoost predicted points: 92.0
  Neural Net predicted points: 91.0



## Step 5 — Lessons learned & future work

### Practical experience — XGBoost vs. the neural network

**XGBoost was the pragmatic choice for a fixed time budget.** The sklearn-compatible API meant no manual training loop, no device/dtype bookkeeping, no gradient instability to chase — the model was either underfit or overfit, never NaN or diverging. Its one real device-detection surprise was that `torch.cuda.is_available()` can't be trusted for it at all — XGBoost bundles its own CUDA runtime independent of whatever torch build happens to be installed, so `detect_xgboost_device()` has to probe XGBoost directly with a throwaway fit. Its other real debugging detour was `shap`'s `TreeExplainer` failing outright against XGBoost 3.x's `base_score` format (bracket-array notation `shap` didn't expect) — required reading XGBoost's actual serialized model format to understand, then a monkeypatch to fix. Beyond those two, XGBoost's biggest win came from feature engineering (adding the TF-IDF block), not from hyperparameter search — ~10 separate Optuna searches on tabular features alone plateaued at test R² ≈ 0.71-0.73 regardless of depth/regularization choices.

**The neural network had far more moving parts, and far more of them broke at least once.** A custom training loop meant owning early-stopping bookkeeping, best-state checkpointing, gradient clipping, and an LR scheduler by hand. `BatchNorm1d` silently crashes on a trailing batch of exactly one sample — a bug that only shows up for specific `(dataset size, batch size)` combinations, not something a quick test would necessarily catch. The single most subtle bug this project hit: `NNModelRegistry` didn't originally persist which activation function a checkpoint was trained with — `nn.ReLU`/`nn.GELU`/`nn.SiLU` have no learnable parameters, so `load_state_dict` succeeds silently even when the wrong one is reconstructed, quietly computing a different function than what was actually trained. It also took real, iterative investment to close most (not all) of the gap to XGBoost — three separate rounds (wider search, activation choice, gradient clipping, an LR scheduler) — where XGBoost needed effectively one big lever (the text features) to reach its final number. The NN's SHAP explainability needed a different explainer (`GradientExplainer`, not `TreeExplainer`) with its own compatibility issue (a flat-output-shape mismatch, fixed with a small wrapper module) — another thing that had to be diagnosed rather than assumed to work.

**Net experiential takeaway**: the performance gap between the two models tracks the engineering-investment gap almost as much as any fundamental capability difference. XGBoost got most of the way there almost for free; the NN's flexibility only started paying off after substantial additional debugging and tuning work that wasn't obvious would be needed up front.

### Practical experience — working with this dataset

Real data-quality issues had to be *worked around*, not just *described* in an EDA notebook: missing `region_2` (61%), missing `price` (7%, needing grouped-median imputation with a global fallback), missing `taster_name` (20%), and a vintage year that only exists as free text buried in a `title` field, extractable via regex for 96.5% of rows. Building a preprocessing pipeline robust to all of that — `DataCleaner` → `FeatureEngineer` → `TabularEncoder` → `TextEncoder`, each with its own fitted state — was real design work, not a one-off `read_csv` and go.

That pipeline also accumulated genuine accidental complexity along the way: an entire second, orphaned `PreprocessingPipeline` class existed in the codebase, unused by any notebook, and had quietly drifted out of sync with the pipeline notebook 02 actually uses — it never passed `include_strictness=True`, so it would have silently produced a dataset missing the `taster_strictness` feature if anyone had ever run it. A real lesson about keeping one source of truth for a data pipeline rather than two parallel implementations that can drift apart unnoticed.

The clearest example of how much implicit knowledge a preprocessing pipeline accumulates: building the "predict a new wine" feature (Step 4 above) required figuring out exactly which parts of `TabularEncoder`'s fitted state are simple per-row transforms, which are training-set aggregates, and which are fit on the *full* dataset versus train-only — not obvious from the outside, only resolved by reading `cleaner.py`/`feature_engineer.py`/`encoders.py` in full. It was only trustworthy once verified against real data: reconstructing a known test-set row through the rebuilt pipeline and diffing it byte-for-byte against the actual persisted training-time feature array (max difference: `0.0`).

### Why train R² is so much higher than val/test

| model | train R² | test R² | gap |
|---|---|---|---|
| XGBoost | 0.878 | 0.775 | 0.103 |
| Neural Net | 0.906 | 0.769 | 0.137 |

This gap isn't an absence of regularization — both models already apply real anti-overfitting measures (XGBoost: `reg_alpha`/`reg_lambda`/`subsample`/`colsample_bytree`/early stopping; NN: dropout up to 0.44, weight decay, BatchNorm, early stopping, gradient clipping). What's left after all of that traces to four more structural causes:

1. **The target-encoded features leak a controlled amount of training-set structure by construction.** `winery_avg_points`, `taster_avg_points`, `variety_avg_points`, and friends are computed *from the training labels themselves* — 5-fold CV during fitting reduces this leakage but can't remove it entirely, especially for low-frequency categories (a winery with 2 training reviews has a `winery_avg_points` that's essentially those 2 rows' average — a tight fit on train, weak generalization to anything else).
2. **Both models have real capacity to fit noise that is, by definition, not reproducible in held-out data.** The taster-bias ANOVA alone shows reviewer identity explains ~10% of the variance in `points` — and that's just the *measurable* part of subjective scoring noise (mood, palate fatigue, and other unmeasured factors almost certainly add more). A high-capacity model (XGBoost: depth up to 8, ~500 trees; NN: up to 1024 hidden units across 3-4 layers) can fit some of that idiosyncratic, non-generalizable noise in the specific training rows it sees, which lowers train error without any matching test improvement.
3. **The TF-IDF block is high-dimensional relative to the row count** (2000 mostly-sparse text columns against 83k training rows) — exactly the regime where a flexible model can fit word-combination quirks specific to the training corpus that don't hold up on new text.
4. **Optuna optimized validation RMSE, not the train-val gap.** The hyperparameter search picks whichever config does best on val — it was never asked to also minimize how much better train does than val, so a wider gap is tolerated whenever it doesn't cost anything on val itself. This is worth naming explicitly: pushing for *more* regularization from here would trade train fit for, at best, a marginal val/test gain — points 1-3 are structural to this feature set and this dataset's labels, not a hyperparameter that was simply left untuned.

### What we'd improve — modeling
- Replace the TF-IDF block with a frozen pretrained sentence-embedding (e.g. `sentence-transformers`) — same "concatenate as another dense block" architecture both models already use, no RNN/Transformer training required, likely a stronger and more efficient text representation than TF-IDF.
- Keep widening the NN's hyperparameter search now that it's cheap (dense preloading + Optuna pruning cut wall-clock substantially) — more trials, per-layer dropout, independently-searched depth/width instead of fixed architecture strings.
- Exclude BatchNorm/bias parameters from weight decay — a well-known best practice not yet applied; AdamW currently regularizes every NN parameter uniformly.
- A cheap ensemble (blend/average XGBoost + NN predictions) — the two models reach similar accuracy through different mechanisms (individual TF-IDF words vs. tabular aggregates, per notebook 05's SHAP comparison), so their errors may be meaningfully decorrelated.

### What we'd improve — data
(from Step 2 above, reframed as concrete next steps)
- More reviewers per wine, averaged, to shrink the ~10%-of-variance taster-bias noise floor directly.
- Consistent sub-region/appellation labels across all countries, not just some US wines, to extend the `region_1`/`province` signal both models already rely on.
- Review timing normalized relative to vintage/release date, to remove the vintage survivorship-bias confound rather than encode it.
- More reviews for underrepresented countries/varieties/wineries specifically — not just more rows overall — to stabilize the target- and frequency-encoded features that are noisiest exactly where data is thinnest.
- A second review-text source, to test whether the text signal generalizes beyond one publication's tasting-note vocabulary.

## Step 6 — Export best results from W&B

Pulls the single best-recorded run per model (the same `best_run_id` already used throughout this notebook) from Weights & Biases via `wandb.Api()` and writes its config + final metrics to `reports/wandb_best_results.csv`.

Also pulls the head-to-head **model-comparison** run (`group="model-comparison"`, logged by notebook 05 — the most recent one if it's been run more than once) and builds a self-contained `reports/wandb_best_results.html`: the exact RMSE/MAE/R² comparison table plus the scorecard, overfitting, predicted-vs-actual, error-by-score-bucket, and residual-overlay charts, all embedded directly (images as base64, the table as real HTML) — viewable by opening the file, no W&B account needed at all.

Matching a `run_id` to its W&B run isn't a straight equality check — the `name=` set at `wandb.init()` time varies by which training step produced it (`run_name`, `f"xgboost-{...}"`, `f"nn-{...}"`, `f"shap-{...}"` all appear across notebooks 03/04). Every one of those patterns ends with the run's own `run_id`, though, so matching on `name.endswith(run_id)` works regardless of which step logged it.

In [8]:
if WANDB_ENABLED:
    import base64
    import re
    import tempfile
    from html import escape as html_escape

    def _fetch_image_b64(
        run: "wandb.apis.public.Run", summary_key: str, tmp_dir: str
    ) -> str | None:
        """Download a wandb-logged image and return it as a base64 PNG data URI."""
        image_meta = run.summary.get(summary_key)
        if not image_meta or "path" not in image_meta:
            return None
        wandb_file = run.file(image_meta["path"])
        wandb_file.download(root=tmp_dir, replace=True)
        image_bytes = (Path(tmp_dir) / wandb_file.name).read_bytes()
        return f"data:image/png;base64,{base64.b64encode(image_bytes).decode()}"

    def _fetch_table(run: "wandb.apis.public.Run", summary_key: str, tmp_dir: str) -> dict | None:
        """Download a wandb-logged Table and return its {columns, data} dict."""
        table_meta = run.summary.get(summary_key)
        if not table_meta or "path" not in table_meta:
            return None
        wandb_file = run.file(table_meta["path"])
        wandb_file.download(root=tmp_dir, replace=True)
        return json.loads((Path(tmp_dir) / wandb_file.name).read_text(encoding="utf-8"))

    def _table_to_html(table: dict) -> str:
        header = "".join(f"<th>{html_escape(str(c))}</th>" for c in table["columns"])
        body_rows = ""
        for row in table["data"]:
            cells_html = "".join(
                f"<td>{v:.4f}</td>" if isinstance(v, float) else f"<td>{html_escape(str(v))}</td>"
                for v in row
            )
            body_rows += f"<tr>{cells_html}</tr>"
        return f"<table><thead><tr>{header}</tr></thead><tbody>{body_rows}</tbody></table>"

    def _inline_md(text: str) -> str:
        """Minimal static-prose converter: **bold**, `code`, *italic* -> HTML."""
        text = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", text)
        text = re.sub(r"`([^`]+)`", r"<code>\1</code>", text)
        text = re.sub(r"\*(.+?)\*", r"<em>\1</em>", text)
        return text

    def _p(text: str) -> str:
        return f"<p>{_inline_md(text)}</p>\n"

    def _li(items: list[str]) -> str:
        return "".join(f"<li>{_inline_md(item)}</li>\n" for item in items)

    def _slide(kicker: str, title: str, body: str) -> str:
        return (
            f'<section class="slide">\n<div class="slide-inner">\n'
            f'<p class="kicker">{html_escape(kicker)}</p>\n<h2>{title}</h2>\n{body}'
            "</div>\n</section>\n"
        )

    api = wandb.Api()
    entity_project = f"{api.default_entity}/{os.environ.get('WANDB_PROJECT', 'diplo-mod-1')}"
    all_runs = list(api.runs(entity_project))

    rows = []
    for label, run_id in [("xgboost", xgb_history.best_run_id), ("nn", nn_history.best_run_id)]:
        matches = [r for r in all_runs if r.name and r.name.endswith(run_id)]
        if not matches:
            print(f"No W&B run found matching '{run_id}' ({label}) -- skipping.")
            continue
        run = matches[0]
        row = {"model": label, "run_id": run_id, "wandb_name": run.name, "wandb_url": run.url}
        row.update({f"config_{k}": v for k, v in run.config.items()})
        row.update(
            {f"summary_{k}": v for k, v in run.summary._json_dict.items() if not k.startswith("_")}
        )
        rows.append(row)
        print(f"{label}: {run.url}")

    # The head-to-head comparison run (notebook 05) isn't tracked in reports/*.json
    # like the training runs are -- find it by group instead, most recent first.
    comparison_matches = list(
        api.runs(entity_project, filters={"group": "model-comparison"}, order="-created_at")
    )
    comparison_run = comparison_matches[0] if comparison_matches else None

    result_slides = ""
    with tempfile.TemporaryDirectory() as tmp_dir:
        if comparison_run is not None:
            print(f"model-comparison: {comparison_run.url}")
            table = _fetch_table(comparison_run, "comparison_table", tmp_dir)
            table_html = _table_to_html(table) if table else "<p>Comparison table not found.</p>"

            result_slides += _slide(
                "Model comparison",
                "RMSE / MAE / R\u00b2 by split",
                f'<p>Run: <a href="{comparison_run.url}">{html_escape(comparison_run.name)}</a></p>\n'
                f'<div class="card">{table_html}</div>\n',
            )

            chart_keys = [
                "scorecard",
                "overfitting_bars",
                "predicted_vs_actual",
                "error_by_points_bucket",
                "residual_overlay",
            ]
            for i, key in enumerate(chart_keys, start=1):
                b64 = _fetch_image_b64(comparison_run, key, tmp_dir)
                if b64:
                    title = html_escape(key.replace("_", " ").title())
                    result_slides += _slide(
                        f"Chart {i} of {len(chart_keys)}",
                        title,
                        f'<img src="{b64}" alt="{title}">\n',
                    )
        else:
            print("No model-comparison run found -- run notebook 05 with WANDB_ENABLED=true first.")

    lessons_slides = (
        _slide(
            "Reflections",
            "XGBoost vs. the neural network",
            _p(
                "**XGBoost was the pragmatic choice for a fixed time budget.** The "
                "sklearn-compatible API meant no manual training loop, no device/dtype "
                "bookkeeping, no gradient instability to chase \u2014 the model was "
                "either underfit or overfit, never NaN or diverging. Its one real "
                "device-detection surprise was that `torch.cuda.is_available()` can't "
                "be trusted for it at all \u2014 XGBoost bundles its own CUDA runtime "
                "independent of whatever torch build happens to be installed, so "
                "`detect_xgboost_device()` has to probe XGBoost directly with a "
                "throwaway fit. Its other real debugging detour was `shap`'s "
                "`TreeExplainer` failing outright against XGBoost 3.x's `base_score` "
                "format \u2014 required reading XGBoost's actual serialized model "
                "format to understand, then a monkeypatch to fix. Beyond those two, "
                "XGBoost's biggest win came from feature engineering (adding the "
                "TF-IDF block), not from hyperparameter search \u2014 ~10 separate "
                "Optuna searches on tabular features alone plateaued at test "
                "R\u00b2 \u2248 0.71-0.73 regardless of depth/regularization choices."
            )
            + _p(
                "**The neural network had far more moving parts, and far more of "
                "them broke at least once.** A custom training loop meant owning "
                "early-stopping bookkeeping, best-state checkpointing, gradient "
                "clipping, and an LR scheduler by hand. `BatchNorm1d` silently "
                "crashes on a trailing batch of exactly one sample \u2014 a bug that "
                "only shows up for specific `(dataset size, batch size)` "
                "combinations. The single most subtle bug this project hit: "
                "`NNModelRegistry` didn't originally persist which activation "
                "function a checkpoint was trained with \u2014 `nn.ReLU`/`nn.GELU`/"
                "`nn.SiLU` have no learnable parameters, so `load_state_dict` "
                "succeeds silently even when the wrong one is reconstructed. It also "
                "took three separate tuning rounds (wider search, activation choice, "
                "gradient clipping, an LR scheduler) to close most of the gap to "
                "XGBoost, where XGBoost needed effectively one big lever (the text "
                "features). The NN's SHAP explainability needed a different "
                "explainer (`GradientExplainer`, not `TreeExplainer`) with its own "
                "compatibility fix, too."
            )
            + _p(
                "**Net experiential takeaway**: the performance gap between the two "
                "models tracks the engineering-investment gap almost as much as any "
                "fundamental capability difference. XGBoost got most of the way "
                "there almost for free; the NN's flexibility only started paying "
                "off after substantial additional debugging and tuning work."
            ),
        )
        + _slide(
            "Reflections",
            "Working with this dataset",
            _p(
                "Real data-quality issues had to be *worked around*, not just "
                "*described* in an EDA notebook: missing `region_2` (61%), missing "
                "`price` (7%, needing grouped-median imputation with a global "
                "fallback), missing `taster_name` (20%), and a vintage year only "
                "extractable via regex from a free-text `title` field (96.5% of "
                "rows). Building a preprocessing pipeline robust to all of that "
                "\u2014 `DataCleaner` \u2192 `FeatureEngineer` \u2192 "
                "`TabularEncoder` \u2192 `TextEncoder`, each with its own fitted "
                "state \u2014 was real design work, not a one-off `read_csv` and go."
            )
            + _p(
                "That pipeline also accumulated genuine accidental complexity: an "
                "entire second, orphaned `PreprocessingPipeline` class existed in "
                "the codebase, unused by any notebook, and had quietly drifted out "
                "of sync with the pipeline notebook 02 actually uses \u2014 it "
                "never passed `include_strictness=True`, so it would have silently "
                "produced a dataset missing the `taster_strictness` feature if "
                "anyone had ever run it. A real lesson about keeping one source of "
                "truth for a data pipeline."
            )
            + _p(
                "The clearest example of how much implicit knowledge a "
                "preprocessing pipeline accumulates: building the \"predict a new "
                "wine\" feature required figuring out exactly which parts of "
                "`TabularEncoder`'s fitted state are per-row transforms, "
                "training-set aggregates, or fit on the *full* dataset versus "
                "train-only \u2014 only resolved by reading the encoder source in "
                "full, and only trustworthy once verified byte-for-byte against "
                "real persisted training data (max difference: `0.0`)."
            ),
        )
        + _slide(
            "Reflections",
            "Why train R\u00b2 is so much higher than val/test",
            '<div class="card"><table><thead><tr>'
            "<th>model</th><th>train R\u00b2</th><th>test R\u00b2</th><th>gap</th>"
            "</tr></thead><tbody>"
            "<tr><td>XGBoost</td><td>0.878</td><td>0.775</td><td>0.103</td></tr>"
            "<tr><td>Neural Net</td><td>0.906</td><td>0.769</td><td>0.137</td></tr>"
            "</tbody></table></div>\n"
            + _p(
                "This gap isn't an absence of regularization \u2014 both models "
                "already apply real anti-overfitting measures. What's left traces "
                "to four more structural causes:"
            )
            + "<ol>\n"
            + _li(
                [
                    "**Target-encoded features leak training-set structure by "
                    "construction.** `winery_avg_points`, `taster_avg_points`, and "
                    "friends are computed *from the training labels themselves* "
                    "\u2014 5-fold CV reduces this leakage but can't remove it, "
                    "especially for low-frequency categories.",
                    "**Both models have capacity to fit noise that isn't "
                    "reproducible in held-out data.** Taster-bias ANOVA alone shows "
                    "reviewer identity explains ~10% of the variance in `points` "
                    "\u2014 and that's just the *measurable* part of subjective "
                    "scoring noise.",
                    "**The TF-IDF block is high-dimensional relative to the row "
                    "count** (2000 mostly-sparse text columns against 83k training "
                    "rows) \u2014 a classic regime for fitting training-corpus-"
                    "specific quirks that don't generalize.",
                    "**Optuna optimized validation RMSE, not the train-val gap.** "
                    "A wider gap is tolerated whenever it doesn't cost anything on "
                    "val itself \u2014 more regularization from here would trade "
                    "train fit for, at best, a marginal val/test gain; points 1-3 "
                    "are structural, not a tuning oversight.",
                ]
            )
            + "</ol>\n",
        )
        + _slide(
            "What we'd improve",
            "Modeling",
            "<ul>\n"
            + _li(
                [
                    "Replace the TF-IDF block with a frozen pretrained "
                    "sentence-embedding (e.g. `sentence-transformers`) \u2014 same "
                    "architecture both models already use, likely stronger than "
                    "TF-IDF.",
                    "Keep widening the NN's hyperparameter search now that it's "
                    "cheap \u2014 more trials, per-layer dropout, independently-"
                    "searched depth/width.",
                    "Exclude BatchNorm/bias parameters from weight decay \u2014 a "
                    "well-known best practice not yet applied.",
                    "A cheap ensemble (blend XGBoost + NN predictions) \u2014 the "
                    "two models reach similar accuracy through different "
                    "mechanisms, so their errors may be meaningfully decorrelated.",
                ]
            )
            + "</ul>\n",
        )
        + _slide(
            "What we'd improve",
            "Data",
            "<ul>\n"
            + _li(
                [
                    "More reviewers per wine, averaged, to shrink the "
                    "~10%-of-variance taster-bias noise floor directly.",
                    "Consistent sub-region/appellation labels across all "
                    "countries, not just some US wines.",
                    "Review timing normalized relative to vintage/release date, to "
                    "remove the vintage survivorship-bias confound.",
                    "More reviews for underrepresented countries/varieties/"
                    "wineries specifically, to stabilize the noisiest "
                    "target-/frequency-encoded features.",
                    "A second review-text source, to test whether the text signal "
                    "generalizes beyond one publication's vocabulary.",
                ]
            )
            + "</ul>\n",
        )
    )

    if rows:
        best_results_df = pd.DataFrame(rows)
        best_results_path = REPORTS / "wandb_best_results.csv"
        best_results_df.to_csv(best_results_path, index=False)
        print(f"Saved {best_results_path}")

        title_slide = (
            '<section class="slide title-slide">\n<div class="slide-inner">\n'
            '<p class="kicker">Weights &amp; Biases export</p>\n'
            "<h1>Best Model Results</h1>\n"
            "<p>diplo-mod-1 &mdash; generated from the best-recorded run per model, "
            "and the head-to-head model-comparison run. Self-contained: images and "
            "tables are embedded below, no W&amp;B account needed to view this "
            "file.</p>\n"
            "</div>\n</section>\n"
        )
        deck_html = title_slide + result_slides + lessons_slides

        CSS = '\n    :root {\n      --bg: #0a0e14;\n      --ink: #eef3fa;\n      --ink-dim: rgba(238, 243, 250, .68);\n      --muted: rgba(238, 243, 250, .45);\n      --accent: #ffb454;\n      --accent2: #59d3ff;\n      --card-bg: rgba(255, 255, 255, .04);\n      --card-border: rgba(255, 255, 255, .10);\n    }\n    * { box-sizing: border-box; }\n    html, body { height: 100%; margin: 0; }\n    body {\n      background: var(--bg);\n      color: var(--ink);\n      font-family: "Inter", system-ui, sans-serif;\n      font-weight: 300;\n      line-height: 1.6;\n      overflow: hidden;\n    }\n    .kicker {\n      font-family: "IBM Plex Mono", ui-monospace, monospace;\n      font-size: .72rem;\n      letter-spacing: .18em;\n      text-transform: uppercase;\n      color: var(--accent2);\n      margin: 0 0 .75rem;\n    }\n    h1 {\n      font-family: "Instrument Serif", Georgia, serif;\n      font-weight: 400;\n      font-size: 3.4rem;\n      line-height: 1.1;\n      margin: 0 0 1.25rem;\n    }\n    h2 {\n      font-family: "Instrument Serif", Georgia, serif;\n      font-weight: 400;\n      font-size: 2.1rem;\n      margin: 0 0 1.25rem;\n    }\n    p { color: var(--ink-dim); font-size: 1rem; max-width: 640px; }\n    a { color: var(--accent); text-decoration: none; border-bottom: 1px solid rgba(255, 180, 84, .35); }\n    a:hover { border-bottom-color: var(--accent); }\n    strong { color: var(--ink); font-weight: 500; }\n    em { font-style: italic; }\n    code {\n      font-family: "IBM Plex Mono", ui-monospace, monospace;\n      font-size: .88em;\n      background: rgba(255, 255, 255, .06);\n      color: var(--accent2);\n      padding: .12em .4em;\n      border-radius: 4px;\n    }\n    ol, ul { max-width: 640px; padding-left: 1.2rem; color: var(--ink-dim); }\n    li { margin-bottom: .6rem; }\n    .card {\n      background: var(--card-bg);\n      border: 1px solid var(--card-border);\n      border-radius: 18px;\n      padding: 1.5rem 1.75rem;\n      margin: 0 0 1.5rem;\n      overflow-x: auto;\n      max-width: 100%;\n    }\n    table { border-collapse: collapse; width: 100%; font-family: "IBM Plex Mono", ui-monospace, monospace; font-size: .85rem; }\n    th, td { padding: .6rem .9rem; text-align: right; border-bottom: 1px solid var(--card-border); white-space: nowrap; }\n    th { color: var(--accent2); font-weight: 500; text-transform: uppercase; letter-spacing: .04em; font-size: .72rem; }\n    td:first-child, td:nth-child(2), th:first-child, th:nth-child(2) { text-align: left; }\n    td:first-child, td:nth-child(2) { color: var(--ink); }\n    tbody tr:hover { background: rgba(255, 255, 255, .03); }\n\n    .deck {\n      display: flex;\n      width: 100%;\n      height: 100vh;\n      overflow-x: auto;\n      overflow-y: hidden;\n      scroll-snap-type: x mandatory;\n      scroll-behavior: smooth;\n    }\n    .slide {\n      min-width: 100vw;\n      height: 100vh;\n      flex-shrink: 0;\n      scroll-snap-align: start;\n      overflow-y: auto;\n      display: flex;\n      align-items: center;\n      background-image:\n        radial-gradient(ellipse 800px 420px at 12% -10%, rgba(255, 180, 84, .10), transparent),\n        radial-gradient(ellipse 800px 420px at 88% 0%, rgba(89, 211, 255, .08), transparent);\n      background-repeat: no-repeat;\n    }\n    .slide-inner { max-width: 880px; margin: 0 auto; padding: 4rem 6vw; width: 100%; }\n    .slide img {\n      max-width: 100%;\n      max-height: 52vh;\n      object-fit: contain;\n      border-radius: 10px;\n      border: 1px solid var(--card-border);\n      display: block;\n    }\n\n    .progress { position: fixed; top: 0; left: 0; width: 100%; height: 3px; background: rgba(255, 255, 255, .06); z-index: 30; }\n    .progress-fill { height: 100%; width: 0%; background: linear-gradient(90deg, var(--accent), var(--accent2)); transition: width .25s ease; }\n    .nav-arrow {\n      position: fixed; top: 50%; transform: translateY(-50%); z-index: 30;\n      width: 44px; height: 44px; border-radius: 50%;\n      background: var(--card-bg); border: 1px solid var(--card-border); color: var(--ink);\n      font-size: 1.1rem; cursor: pointer;\n    }\n    .nav-arrow:hover { background: rgba(255, 255, 255, .08); }\n    .nav-arrow:disabled { opacity: .2; cursor: default; }\n    .nav-prev { left: 1.25rem; }\n    .nav-next { right: 1.25rem; }\n    .dots { position: fixed; bottom: 1.5rem; left: 50%; transform: translateX(-50%); display: flex; gap: .5rem; z-index: 30; }\n    .dot { width: 8px; height: 8px; border-radius: 50%; border: none; background: var(--card-border); cursor: pointer; padding: 0; transition: all .2s ease; }\n    .dot.active { background: var(--accent); width: 22px; border-radius: 4px; }\n    .counter {\n      position: fixed; top: 1.25rem; right: 1.5rem; z-index: 30;\n      font-family: "IBM Plex Mono", ui-monospace, monospace; font-size: .75rem; color: var(--muted);\n    }\n    .brand-tag {\n      position: fixed; bottom: 1.5rem; left: 1.5rem; z-index: 30;\n      font-family: "IBM Plex Mono", ui-monospace, monospace; font-size: .7rem; color: var(--muted);\n      letter-spacing: .06em; text-transform: uppercase;\n    }\n'
        JS = "\n    (function () {\n      var deck = document.getElementById('deck');\n      var slides = Array.prototype.slice.call(deck.children);\n      var dotsEl = document.getElementById('dots');\n      var progressFill = document.getElementById('progressFill');\n      var prevBtn = document.getElementById('prevBtn');\n      var nextBtn = document.getElementById('nextBtn');\n      var counterEl = document.getElementById('counter');\n\n      slides.forEach(function (_, i) {\n        var dot = document.createElement('button');\n        dot.className = 'dot';\n        dot.setAttribute('aria-label', 'Go to slide ' + (i + 1));\n        dot.addEventListener('click', function () { goTo(i); });\n        dotsEl.appendChild(dot);\n      });\n      var dots = Array.prototype.slice.call(dotsEl.children);\n\n      function currentIndex() {\n        var idx = Math.round(deck.scrollLeft / deck.clientWidth);\n        return Math.max(0, Math.min(slides.length - 1, idx));\n      }\n\n      function goTo(idx) {\n        idx = Math.max(0, Math.min(slides.length - 1, idx));\n        slides[idx].scrollIntoView({ behavior: 'smooth', inline: 'start', block: 'nearest' });\n      }\n\n      function updateNav() {\n        var idx = currentIndex();\n        dots.forEach(function (d, i) { d.classList.toggle('active', i === idx); });\n        progressFill.style.width = ((idx + 1) / slides.length * 100) + '%';\n        prevBtn.disabled = idx === 0;\n        nextBtn.disabled = idx === slides.length - 1;\n        counterEl.textContent = (idx + 1) + ' / ' + slides.length;\n      }\n\n      var scrollTimer = null;\n      deck.addEventListener('scroll', function () {\n        if (scrollTimer) window.cancelAnimationFrame(scrollTimer);\n        scrollTimer = window.requestAnimationFrame(updateNav);\n      });\n      prevBtn.addEventListener('click', function () { goTo(currentIndex() - 1); });\n      nextBtn.addEventListener('click', function () { goTo(currentIndex() + 1); });\n      document.addEventListener('keydown', function (e) {\n        if (e.key === 'ArrowRight' || e.key === 'PageDown') goTo(currentIndex() + 1);\n        if (e.key === 'ArrowLeft' || e.key === 'PageUp') goTo(currentIndex() - 1);\n      });\n      // Plain mouse wheels only scroll vertically -- redirect that into\n      // horizontal deck movement, but only when the active slide's own\n      // content fits in one viewport, so long prose slides stay\n      // wheel-scrollable within themselves.\n      deck.addEventListener('wheel', function (e) {\n        var active = slides[currentIndex()];\n        var canScrollWithin = active.scrollHeight > active.clientHeight + 1;\n        if (!canScrollWithin && Math.abs(e.deltaY) > Math.abs(e.deltaX)) {\n          deck.scrollLeft += e.deltaY;\n          e.preventDefault();\n        }\n      }, { passive: false });\n\n      updateNav();\n    })();\n"

        html_report = (
            '<!doctype html>\n<html lang="en">\n<head>\n<meta charset="utf-8">\n'
            "<title>diplo-mod-1 -- Best Model Results</title>\n"
            '<link rel="preconnect" href="https://fonts.googleapis.com">\n'
            '<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>\n'
            '<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500;600&family=Inter:wght@300;400;500;600&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">\n'
            "<style>" + CSS + "</style>\n</head>\n<body>\n"
            '<div class="progress"><div class="progress-fill" id="progressFill"></div></div>\n'
            '<div class="deck" id="deck">\n' + deck_html + "</div>\n"
            '<button class="nav-arrow nav-prev" id="prevBtn" aria-label="Previous slide">&#8592;</button>\n'
            '<button class="nav-arrow nav-next" id="nextBtn" aria-label="Next slide">&#8594;</button>\n'
            '<div class="dots" id="dots"></div>\n'
            '<div class="counter" id="counter"></div>\n'
            '<div class="brand-tag">diplo-mod-1</div>\n'
            "<script>" + JS + "</script>\n</body>\n</html>"
        )
        html_path = REPORTS / "wandb_best_results.html"
        html_path.write_text(html_report, encoding="utf-8")
        print(f"Saved {html_path}")
else:
    print("WANDB_ENABLED is false in .env -- set it to true to export W&B results.")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


xgboost: https://wandb.ai/leonardo-a-heis/diplo-mod-1/runs/1xg4h2rj
nn: https://wandb.ai/leonardo-a-heis/diplo-mod-1/runs/75rbilv8
model-comparison: https://wandb.ai/leonardo-a-heis/diplo-mod-1/runs/2p4857gp
Saved ..\reports\wandb_best_results.csv
Saved ..\reports\wandb_best_results.html
